## 1. Contexto del Avance


En este avance, se automatiza la transición de FleetLogix desde una infraestructura local limitada hacia una Arquitectura Cloud Escalable en AWS. Resuelve la fragmentación de datos y la falta de disponibilidad al migrar los activos maestros (conductores y vehículos) a Amazon RDS, mientras habilita un cerebro operativo mediante AWS Lambda y DynamoDB para procesar telemetría en tiempo real. Pasamos de un sistema estático a un ecosistema Event-Driven capaz de reaccionar a desvíos de ruta y calcular tiempos de llegada (ETA) de forma automática.

##  Configuración de Infraestructura 
Se utiliza un script de automatización (aws_setup.py) para crear un entorno aislado.

Recursos: Creación de RDS (PostgreSQL) para datos relacionales, DynamoDB para baja latencia y S3 para el Data Lake de logs históricos.

Seguridad: Implementación de IAM Roles bajo el principio de menor privilegio, asegurando que las funciones Lambda solo accedan a lo estrictamente necesario.

Sección II: Lógica de Negocio en la Nube (AWS Lambda)
Se realiza el desacoplamiento de funciones. Se crearon tres microservicios:

verificar-entrega: Validación de estados en DynamoDB.

calcular_eta: Procesamiento matemático de telemetría.

alerta_desvio: Monitoreo de seguridad perimetral (Geofencing).

Tecnología: Uso de Boto3 y Variables de Entorno para garantizar que el código sea portátil y seguro.

Sección III: Migración Híbrida (ETL)
Uso de psycopg2 para conectar el mundo local con la nube.
Ejecutamos una extracción desde el PostgreSQL local y carga en el Endpoint de RDS. Esto garantiza que la "Fuente de Verdad" ahora resida en un entorno administrado con backups automáticos.

3. Resultados y Validación 
No se realizo la validacion mediante la ejecucion del proyecto migratorio a la nube dado a qu aws restringui la ingesta de datos a el plan gratuito, pero si se comprendio y diagramo el proceso de cada una de las funciones argumntando el conocimiento adquirido en el archivo PDF, AWS_Analisis_Arquitetura.pdf, y en la explicacion del diagrama de arquitetura AWS contenidos dentro de esta carpeta Avance_4, como asi tambien armando la estructura en la nube siguiendo la metodologia del documento "entregables_y_paso_a_paso_AWS.pdf"



## Conclusión General del Análisis de Arquitectura
El presente análisis demuestra que la transición de FleetLogix hacia un ecosistema Cloud no es solo un cambio de almacenamiento, sino una evolución en la eficiencia operativa. Al diseñar esta arquitectura, se han tomado decisiones estratégicas que resuelven los retos de la logística moderna:

Desacoplamiento Operativo: Se logró separar la "información maestra" (en Amazon RDS) de la "información en tiempo real" (en DynamoDB). Esto garantiza que el sistema sea extremadamente rápido para el conductor y el cliente, pero mantenga la integridad legal y administrativa en la base de datos relacional.

Eficiencia de Costos y Escalabilidad: Al proponer un modelo Serverless con AWS Lambda, el diseño asegura que FleetLogix solo "pague por lo que usa". No es necesario mantener servidores encendidos 24/7; la nube reacciona solo cuando hay un evento (un GPS o una entrega), permitiendo que la flota crezca de 10 a 1,000 camiones sin cambiar el código.

Seguridad y Resiliencia: La arquitectura elimina el riesgo de pérdida de datos por fallos en hardware local. Mediante el uso de S3 para auditoría y las políticas de IAM, el sistema es robusto frente a errores humanos y fallos físicos, asegurando que la historia de cada entrega esté siempre protegida.

En resumen: Aunque este avance se centra en el diseño teórico y analítico, la estructura planteada en los scripts y el diagrama de flechas constituye un plano técnico sólido y profesional. FleetLogix queda proyectada como una plataforma lista para implementar analítica avanzada y monitoreo global en tiempo real.

## Documentación de Implementación y Pruebas (AWS)
# Validación de Infraestructura como Código (IaC)
Se ejecutó el script de automatización aws_setup.py utilizando Python 3.11 y AWS CLI para desplegar los recursos necesarios en la nube. Los resultados obtenidos fueron los siguientes:

Amazon DynamoDB: Se crearon exitosamente cuatro tablas clave para el funcionamiento del sistema: deliveries_status, vehicle_tracking, routes_waypoints y alerts_history.

Amazon S3: Se validó la existencia del bucket de almacenamiento para reportes logísticos.

IAM (Identity and Access Management): Se generó el rol FleetLogixLambdaRole, proporcionando los permisos necesarios para que las funciones Lambda interactúen con otros servicios de AWS.

Gestión de Restricciones (RDS): Durante la creación de la base de datos PostgreSQL en RDS, el sistema arrojó un error de tipo FreeTierRestrictionError. Esto validó que el script cuenta con controles de seguridad que impiden la creación de recursos que excedan los límites de la Capa Gratuita de AWS, protegiendo el presupuesto del proyecto.

# Pruebas de Funciones Serverless (AWS Lambda)
Se realizaron pruebas unitarias sobre la función principal de rastreo para validar la lógica de negocio y la conectividad.

A. Depuración de Código y Sintaxis
Durante el despliegue inicial, se identificaron y corrigieron errores críticos de ejecución:

Error de Indentación (UserCodeSyntaxError): Se ajustó la estructura del código Python siguiendo el estándar PEP 8 (4 espacios), eliminando conflictos de tabulación que impedían el inicio de la función.

Gestión de Dependencias (ImportModuleError): Se corrigió la importación de la librería decimal, asegurando el uso correcto de la clase Decimal para el manejo de coordenadas y precisión numérica.

B. Validación de Conectividad (End-to-End)
Tras el despliegue exitoso del código, se ejecutó un evento de prueba (TestEntrega). El resultado fue Succeeded con una respuesta controlada:

Response: {"statusCode": 404, "body": "{\"error\": \"Entrega no encontrada\"}"}

# Conclusión técnica de la prueba:
Este resultado confirma que:

La función Lambda tiene conectividad total con la base de datos DynamoDB.

La lógica de manejo de excepciones funciona correctamente al buscar un ID que aún no existe en la tabla recién creada.

La arquitectura está lista para recibir datos en tiempo real de los dispositivos GPS.


# Evidencia de Seguridad y Control de Acceso
Se verificó que el acceso a la infraestructura está protegido mediante llaves criptográficas (Access Keys). El sistema denegó intentos de modificación no autorizados cuando las credenciales no estaban presentes (Unable to locate credentials), garantizando que solo el administrador de FleetLogix puede alterar los recursos de la nube.

